In [1]:
#Étape 11 — Étendre la collecte aux cinq actions et à SOXX
import yfinance as yf

tickers = ["AMD", "NVDA", "INTC", "AVGO", "MRVL", "SOXX"]

cours = yf.download(
    tickers,
    start="2020-12-31",
    end="2026-01-01",
    interval="1d",
    auto_adjust=False,
    group_by="column",
    progress=False,
)

clotures = cours["Adj Close"][tickers].sort_index()

display(clotures.loc[["2020-12-31", "2025-12-31"]])
print("Nombre de séances récupérées :", len(clotures))
print("Valeurs manquantes par ticker :", clotures.isna().sum().to_dict())

Ticker,AMD,NVDA,INTC,AVGO,MRVL,SOXX
Date,,,,,,
2020-12-31,91.709999,12.987999,45.038536,39.173416,46.550236,40.337971
2025-12-31,214.160004,186.064606,36.900002,344.202972,84.836945,100.208389


Nombre de séances récupérées : 1256
Valeurs manquantes par ticker : {'AMD': 0, 'NVDA': 0, 'INTC': 0, 'AVGO': 0, 'MRVL': 0, 'SOXX': 0}


In [2]:
#Étape 12 — Enregistrer les données brutes
from pathlib import Path

dossier = Path("data/raw")
dossier.mkdir(parents=True, exist_ok=True)

for ticker in tickers:
    fichier = dossier / f"cours_{ticker}_2021_2025.csv"
    cours.xs(ticker, axis=1, level="Ticker").to_csv(fichier, index_label="Date")
    print(fichier)

print("Dossier des CSV :", dossier.resolve())

data/raw/cours_AMD_2021_2025.csv
data/raw/cours_NVDA_2021_2025.csv
data/raw/cours_INTC_2021_2025.csv
data/raw/cours_AVGO_2021_2025.csv
data/raw/cours_MRVL_2021_2025.csv
data/raw/cours_SOXX_2021_2025.csv
Dossier des CSV : /home/5f91d6e8-26b6-44f2-9b5d-2a89d568f71b/select-equity/data/raw


In [3]:
import pandas as pd

amd_brut = pd.read_csv("data/raw/cours_AMD_2021_2025.csv")
display(amd_brut.head(2))

,Date,Adj Close,Close,High,Low,Open,Volume
0,2020-12-31,91.709999,91.709999,92.300003,90.870003,92.099998,24930700
1,2021-01-04,92.300003,92.300003,96.059998,90.919998,92.110001,51802600


In [4]:
#Étape 13 — Réunir les six CSV dans une seule table
import pandas as pd

tables = []

for ticker in tickers:
    fichier = f"data/raw/cours_{ticker}_2021_2025.csv"
    table = pd.read_csv(fichier, parse_dates=["Date"])
    table["Ticker"] = ticker
    tables.append(table)

cours_journaliers = (
    pd.concat(tables, ignore_index=True)
    .rename(columns={"Adj Close": "Cloture_ajustee", "Close": "Cloture"})
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

display(cours_journaliers[["Date", "Ticker", "Cloture", "Cloture_ajustee"]].head(6))
print("Nombre de lignes :", len(cours_journaliers))
print("Doublons Date–Ticker :", cours_journaliers.duplicated(["Date", "Ticker"]).sum())

,Date,Ticker,Cloture,Cloture_ajustee
0,2020-12-31,AMD,91.709999,91.709999
1,2020-12-31,AVGO,43.785000,39.173416
2,2020-12-31,INTC,49.820000,45.038536
3,2020-12-31,MRVL,47.540001,46.550236
4,2020-12-31,NVDA,13.055000,12.987999
5,2020-12-31,SOXX,42.131111,40.337971


Nombre de lignes : 7536
Doublons Date–Ticker : 0


In [5]:
#Étape 14 — Enregistrer la table préparée

from pathlib import Path

fichier = Path("data/processed/cours_journaliers_2021_2025.csv")
fichier.parent.mkdir(parents=True, exist_ok=True)

cours_journaliers.to_csv(fichier, index=False, date_format="%Y-%m-%d")

controle = pd.read_csv(fichier)
print("Fichier :", fichier)
print("Lignes relues :", len(controle))
print("Colonnes :", controle.columns.tolist())

Fichier : data/processed/cours_journaliers_2021_2025.csv
Lignes relues : 7536
Colonnes : ['Date', 'Cloture_ajustee', 'Cloture', 'High', 'Low', 'Open', 'Volume', 'Ticker']


pivot() : aligne les six séries par date pour appliquer le même calcul à chaque ticker. On soustrait ensuite le rendement de SOXX à chaque rendement pour mesurer l’écart en points de pourcentage.

In [6]:
#Étape 15 — Calculer la performance relative des cinq actions
cours_prepares = pd.read_csv(fichier, parse_dates=["Date"])

clotures = cours_prepares.pivot(
    index="Date",
    columns="Ticker",
    values="Cloture_ajustee",
)

rendement = (
    clotures.loc["2025-12-31"] / clotures.loc["2020-12-31"] - 1
) * 100

comparatif = pd.DataFrame({
    "Rendement cumulé (%)": rendement,
    "Écart vs SOXX (points)": rendement - rendement["SOXX"],
})

display(comparatif.round(2))

,Rendement cumulé (%),Écart vs SOXX (points)
Ticker,,
AMD,133.52,-14.90
AVGO,778.66,630.24
INTC,-18.07,-166.49
MRVL,82.25,-66.17
NVDA,1332.59,1184.17
SOXX,148.42,0.00


pivot() aligne les six séries par date pour appliquer le même calcul à chaque ticker. On soustrait ensuite le rendement de SOXX à chaque rendement pour mesurer l’écart en points de pourcentage.

AMD a gagné 133,52 % sur 2021–2025, soit 14,90 points de moins que SOXX. Sur la seule année 2025, le test pilote montrait au contraire une surperformance de 36,56 points. La période choisie change donc la conclusion.

In [7]:
#Étape 16 — Comparer la perte maximale des six séries
sommets = clotures.cummax()
baisses = clotures / sommets - 1
baisses_2021_2025 = baisses.loc["2021":"2025"]

risque = pd.DataFrame({
    "Perte maximale (%)": baisses_2021_2025.min() * 100,
    "Date du creux": baisses_2021_2025.idxmin().dt.strftime("%d/%m/%Y"),
})

display(risque.round(2))

,Perte maximale (%),Date du creux
Ticker,,
AMD,-65.45,14/10/2022
AVGO,-41.15,04/04/2025
INTC,-70.80,08/04/2025
MRVL,-61.88,05/01/2023
NVDA,-66.34,14/10/2022
SOXX,-45.75,14/10/2022


cummax() retrouve le plus haut cours atteint jusqu’à chaque séance, en incluant la clôture de référence du 31/12/2020. On mesure ensuite à quel point le cours est descendu sous ce sommet. On peut ainsi voir si les actions qui ont le plus progressé sur cinq ans ont aussi connu des baisses plus profondes.

Entre 2021 et 2025, NVIDIA a le meilleur rendement cumulé (+1 332,59 %), avec une baisse maximale de −66,34 % en cours de période. Le rendement final et la perte maximale décrivent deux aspects différents du parcours.

In [8]:
#Étape 17 — Enregistrer la synthèse performance et risque
synthese_marche = comparatif.join(risque).reset_index()

fichier_synthese = "data/processed/synthese_marche_2021_2025.csv"
synthese_marche.to_csv(fichier_synthese, index=False)

controle_synthese = pd.read_csv(fichier_synthese)
display(controle_synthese.round(2))
print("Lignes relues :", len(controle_synthese))

,Ticker,Rendement cumulé (%),Écart vs SOXX (points),Perte maximale (%),Date du creux
0,AMD,133.52,-14.90,-65.45,14/10/2022
1,AVGO,778.66,630.24,-41.15,04/04/2025
2,INTC,-18.07,-166.49,-70.80,08/04/2025
3,MRVL,82.25,-66.17,-61.88,05/01/2023
4,NVDA,1332.59,1184.17,-66.34,14/10/2022
5,SOXX,148.42,0.00,-45.75,14/10/2022


Lignes relues : 6
